# Live RF Model Evaluation Against Current Market Data

This notebook evaluates the Random Forest volatility model against the most recent available market data.

There are two parts:
1. Compare archived live prediction runs across target dates.
2. Evaluate a completed 20-trading-day window where actual future volatility is now known.

## Important: predictions for today/tomorrow cannot be fully judged until 20 future trading days have passed.


# Imports

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Paths

In [2]:
PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

RF_MODELING_PATH = PROJECT_ROOT / "data" / "processed" / "modeling" / "random_forest"
LIVE_PREDICTIONS_PATH = RF_MODELING_PATH / "live_predictions"
LIVE_EVALUATION_PATH = RF_MODELING_PATH / "live_evaluation"

prediction_log_path = LIVE_PREDICTIONS_PATH / "prediction_log.csv"
evaluation_path = LIVE_EVALUATION_PATH / "latest_20d_rf_evaluation.csv"
summary_path = LIVE_EVALUATION_PATH / "latest_20d_rf_evaluation.summary.csv"

prediction_log_path, evaluation_path, summary_path

(WindowsPath('C:/Users/sshab/Documents/coding_projects/INFO442-Group-Project/data/processed/modeling/random_forest/live_predictions/prediction_log.csv'),
 WindowsPath('C:/Users/sshab/Documents/coding_projects/INFO442-Group-Project/data/processed/modeling/random_forest/live_evaluation/latest_20d_rf_evaluation.csv'),
 WindowsPath('C:/Users/sshab/Documents/coding_projects/INFO442-Group-Project/data/processed/modeling/random_forest/live_evaluation/latest_20d_rf_evaluation.summary.csv'))

# Load Archived Live Predictions

The prediction log stores each daily prediction run in long format:
- `Date`: target prediction date
- `ticker`: asset symbol
- `predicted_future_volatility_20d`: model forecast for future 20-day volatility.

In [3]:
prediction_log = pd.read_csv(prediction_log_path, parse_dates=['Date'])

In [4]:
prediction_log.head()

,Date,ticker,predicted_future_volatility_20d
0,2026-08-05,AAPL,0.018918
1,2026-08-05,AGG,0.002464
2,2026-08-05,AMZN,0.018553
3,2026-08-05,CAT,0.017330
4,2026-08-05,GLD,0.014319


# Data Summary

In [5]:
prediction_log_summary = (
    prediction_log
    .groupby("Date")
    .agg(
        tickers=("ticker", "nunique"),
        mean_predicted_volatility=("predicted_future_volatility_20d", "mean"),
        min_predicted_volatility=("predicted_future_volatility_20d", "min"),
        max_predicted_volatility=("predicted_future_volatility_20d", "max"),
    )
    .reset_index()
)

In [6]:
prediction_log_summary

,Date,tickers,mean_predicted_volatility,min_predicted_volatility,max_predicted_volatility
0,2026-08-05,21,0.014474,0.002464,0.018918
1,2026-08-06,21,0.014178,0.002270,0.018328
2,2026-08-10,21,0.014108,0.002461,0.019012


# Compare Prediction Runs Across Days

This chart shows how the model's predicted volatility changed between archived target dates.

In [7]:
fig = px.line(
    prediction_log,
    x="Date",
    y="predicted_future_volatility_20d",
    color="ticker",
    markers=True,
    title="Predicted 20-Day Volatility by Ticker Across Live Runs",
)

fig.update_layout(
    xaxis_title="Prediction Target Date",
    yaxis_title="Predicted Future 20-Day Volatility",
    legend_title="Ticker",
)

fig.show()

In [8]:
latest_two_dates = sorted(prediction_log["Date"].unique())[-2:]

prediction_change = (
    prediction_log[prediction_log["Date"].isin(latest_two_dates)]
    .pivot(index="ticker", columns="Date", values="predicted_future_volatility_20d")
)

prediction_change["change"] = prediction_change.iloc[:, -1] - prediction_change.iloc[:, -2]
prediction_change["absolute_change"] = prediction_change["change"].abs()

prediction_change = prediction_change.sort_values("absolute_change", ascending=False)

prediction_change.head(10)

Date,2026-08-06 00:00:00,2026-08-10 00:00:00,change,absolute_change
ticker,,,,
CAT,0.017001,0.018973,0.001973,0.001973
MSFT,0.017866,0.019012,0.001146,0.001146
JPM,0.015071,0.014018,-0.001053,0.001053
AAPL,0.017746,0.016741,-0.001005,0.001005
PG,0.014053,0.013446,-0.000607,0.000607
VZ,0.016458,0.015904,-0.000554,0.000554
KO,0.016096,0.015593,-0.000503,0.000503
LLY,0.016628,0.016202,-0.000426,0.000426
VNQ,0.008357,0.008781,0.000423,0.000423


In [9]:
fig = px.bar(
    prediction_change.reset_index(),
    x="ticker",
    y="change",
    title="Change in Predicted Volatility Between Latest Two Runs",
)

fig.update_layout(
    xaxis_title="Ticker",
    yaxis_title="Prediction Change",
)

fig.show()

# Load Completed 20-Day Evaluation

This file evaluates a historical prediction date where the next 20 trading days have already happened.

That lets us compare:

`predicted_future_volatility_20d` vs. `actual_future_volatility_20d`

In [10]:
evaluation = pd.read_csv(
    evaluation_path,
    parse_dates=["Date", "future_window_start", "future_window_end"],
)

In [11]:
summary = pd.read_csv(summary_path)

In [12]:
summary

,evaluation_feature_date,horizon_trading_days,tickers,latest_market_date_in_snapshot,mean_future_window_start,mean_future_window_end,MAE,RMSE,R2
0,2026-07-10,20,21,2026-08-07,2026-07-13,2026-08-07,0.005068,0.008277,0.249049


In [13]:
evaluation.head()

,Date,ticker,future_window_start,future_window_end,predicted_future_volatility_20d,actual_future_volatility_20d,error,absolute_error,squared_error
0,2026-07-10,AAPL,2026-07-13,2026-08-07,0.015417,0.023570,0.008153,0.008153,6.647405e-05
1,2026-07-10,AGG,2026-07-13,2026-08-07,0.002361,0.002436,0.000075,0.000075,5.568775e-09
2,2026-07-10,AMZN,2026-07-13,2026-08-07,0.015721,0.040713,0.024993,0.024993,6.246371e-04
3,2026-07-10,CAT,2026-07-13,2026-08-07,0.024674,0.027903,0.003229,0.003229,1.042611e-05
4,2026-07-10,GLD,2026-07-13,2026-08-07,0.015297,0.016505,0.001208,0.001208,1.459354e-06


In [14]:
mae = mean_absolute_error(
    evaluation["actual_future_volatility_20d"],
    evaluation["predicted_future_volatility_20d"],
)

rmse = np.sqrt(
    mean_squared_error(
        evaluation["actual_future_volatility_20d"],
        evaluation["predicted_future_volatility_20d"],
    )
)

r2 = r2_score(
    evaluation["actual_future_volatility_20d"],
    evaluation["predicted_future_volatility_20d"],
)

pd.DataFrame(
    {
        "metric": ["MAE", "RMSE", "R2"],
        "value": [mae, rmse, r2],
    }
)

,metric,value
0,MAE,0.005068
1,RMSE,0.008277
2,R2,0.249049


# Predicted vs Actual Volatility

A perfect model would put every point on the diagonal line.

In [15]:
max_vol = max(
    evaluation["actual_future_volatility_20d"].max(),
    evaluation["predicted_future_volatility_20d"].max(),
)

fig = px.scatter(
    evaluation,
    x="predicted_future_volatility_20d",
    y="actual_future_volatility_20d",
    text="ticker",
    title="Predicted vs Actual Future 20-Day Volatility",
)

fig.add_trace(
    go.Scatter(
        x=[0, max_vol],
        y=[0, max_vol],
        mode="lines",
        name="Perfect Prediction",
        line=dict(dash="dash"),
    )
)

fig.update_traces(textposition="top center")

fig.update_layout(
    xaxis_title="Predicted Future 20-Day Volatility",
    yaxis_title="Actual Future 20-Day Volatility",
)

fig.show()

# Per-Ticker Prediction Error

This shows where the model overpredicted or underpredicted volatility.

In [16]:
evaluation_sorted = evaluation.sort_values("error")

fig = px.bar(
    evaluation_sorted,
    x="ticker",
    y="error",
    title="Prediction Error by Ticker",
)

fig.update_layout(
    xaxis_title="Ticker",
    yaxis_title="Actual - Predicted Volatility",
)

fig.show()

In [17]:
fig = px.bar(
    evaluation.sort_values("absolute_error", ascending=False),
    x="ticker",
    y="absolute_error",
    title="Absolute Prediction Error by Ticker",
)

fig.update_layout(
    xaxis_title="Ticker",
    yaxis_title="Absolute Error",
)

fig.show()

# Predicted and Actual Volatility Side by Side

In [18]:
comparison_long = evaluation.melt(
    id_vars=["ticker"],
    value_vars=[
        "predicted_future_volatility_20d",
        "actual_future_volatility_20d",
    ],
    var_name="volatility_type",
    value_name="volatility",
)

comparison_long["volatility_type"] = comparison_long["volatility_type"].replace(
    {
        "predicted_future_volatility_20d": "Predicted",
        "actual_future_volatility_20d": "Actual",
    }
)

fig = px.bar(
    comparison_long,
    x="ticker",
    y="volatility",
    color="volatility_type",
    barmode="group",
    title="Predicted vs Actual Future 20-Day Volatility by Ticker",
)

fig.update_layout(
    xaxis_title="Ticker",
    yaxis_title="20-Day Volatility",
    legend_title="Type",
)

fig.show()

# Biggest Misses

These are the tickers where the model was furthest from realized volatility.

In [19]:
biggest_misses = (
    evaluation
    .sort_values("absolute_error", ascending=False)
    [
        [
            "ticker",
            "predicted_future_volatility_20d",
            "actual_future_volatility_20d",
            "error",
            "absolute_error",
        ]
    ]
)

In [20]:
biggest_misses.head(10)

,ticker,predicted_future_volatility_20d,actual_future_volatility_20d,error,absolute_error
2,AMZN,0.015721,0.040713,0.024993,0.024993
9,MSFT,0.015987,0.038500,0.022512,0.022512
8,LMT,0.016107,0.025963,0.009856,0.009856
0,AAPL,0.015417,0.023570,0.008153,0.008153
7,LLY,0.016209,0.021425,0.005216,0.005216
6,KO,0.013242,0.018355,0.005113,0.005113
17,VNQ,0.012746,0.008492,-0.004254,0.004254
10,NEE,0.012882,0.008929,-0.003953,0.003953
11,PG,0.015892,0.012247,-0.003645,0.003645
3,CAT,0.024674,0.027903,0.003229,0.003229


# Conclusion

This notebook extends the Random Forest volatility model from historical test-set evaluation to a more realistic live-data setting. We now have archived live prediction runs for the target dates 2026-08-05, 2026-08-06, and 2026-08-10. These newest predictions cannot be fully judged immediately because the model forecasts future 20-day volatility, meaning the actual outcome is only known after 20 trading days have passed.

For the newest same-day baseline comparison, the feature date was 2026-08-07 and the prediction target date was 2026-08-10. The model's mean predicted future volatility was 0.01411, compared with mean trailing 20-day volatility of 0.01733. The correlation between RF predictions and trailing volatility was 0.802, so the model is directionally aligned with recent volatility, but it is forecasting lower volatility overall than the trailing baseline.

To evaluate against real completed market data, the notebook used the latest available feature date with a full 20-trading-day future window. The evaluation used feature rows from 2026-07-10 and compared predictions against realized volatility from 2026-07-13 through 2026-08-07. Across 21 tickers, the model achieved an MAE of 0.00507, RMSE of 0.00828, and R2 of 0.249.

Overall, the Random Forest still shows useful predictive signal on real market data, but it is not perfect. It was closest for assets such as AGG, UNH, QQQ, and XOM, while the largest misses were AMZN and MSFT, where actual realized volatility was much higher than the model forecast. This suggests the model captures general volatility structure, but it can underreact to sudden single-stock volatility spikes.

The main takeaway is that the model is directionally useful for estimating future risk as part of a portfolio analysis workflow, but it should not be treated as an exact forecast or standalone trading signal. Future improvements could include collecting more daily live prediction runs, tracking rolling completed-window evaluations, retraining the model with newer data, and adding event-based features such as earnings dates or market news.